## Workshop - Deconstructing Language for AI Agents

Objective: By the end of this 90-minute workshop, you will understand and have implemented the core components of a Natural Language Processing (NLP) pipeline. You will learn how raw text is systematically processed to become useful, structured data for training an AI agent like Google's BARD.

Duration: 90 minutes

### 1. Introduction: How Does an AI Learn Language?

When you ask an LLM, such as BARD, a question, it seems to just understand you. But that "understanding" is the result of a massive amount of pre-processing.

An AI doesn't read text like a human. It sees text as a sequence of numbers. To an AI, the words "run," "running," and "ran" are initially just different, unrelated data points. The goal of an NLP pipeline is to standardize and simplify text, transforming it into a structured format that a machine can learn patterns from.

Think of it as preparing ingredients for a recipe. You don't just throw a whole carrot into the pot. You wash it, peel it, and chop it. Our "ingredients" are words, and the "recipe" is the AI model we want to train. 

During this workshop, we'll learn the essential pre-processing steps.

### 2. The Raw Material: A Look at Our Text Data

For any NLP task, we need data. We'll be using the "20 Newsgroups" dataset, a classic collection of about 18,000 documents posted to 20 different Usenet newsgroups. It's perfect for us because it's real, messy text with a large vocabulary.

Let's start by loading the data and performing a quick Exploratory Data Analysis (EDA) to understand what we're working with.

Sample EDA: A First Glance at the Data
First, we need to install the necessary libraries. scikit-learn provides the dataset, nltk is our main NLP toolkit, and matplotlib helps us visualize our findings.

In [43]:
#
# Prerequisite installations for the workshop
# To set up the environment, run this script:
# !pip install scikit-learn nltk matplotlib
#
import nltk
import re
import os

# Warning: This download will copy files to your home directory.
# For example, on Linux, it will copy files to ~/.nltk_data.
# In Windows, it will copy files to C:\Users\YourAccount\AppData\Roaming
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

# A better way to handle the download is to:
# Ensure 'punkt' is available and nltk_data path is set
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')
print("Downloading 'punkt' tokenizer...")
nltk.download('punkt', download_dir=nltk_data_path, force=True)
print("Downloading 'punkt_tab' tokenizer...")
nltk.download('punkt_tab', download_dir=nltk_data_path, force=True)
print("Downloading 'stopwords' tokenizer...")
nltk.download('stopwords', download_dir=nltk_data_path, force=True)
print("Downloading 'wordnet' tokenizer...")
nltk.download('wordnet', download_dir=nltk_data_path, force=True)

# Always append the custom nltk_data path (if not already present)
if nltk_data_path not in nltk.data.path:
    nltk.data.path.append(nltk_data_path)

# Debugging paths and contents
print("NLTK Data Paths:", nltk.data.path)
print("Contents of nltk_data:", os.listdir(nltk_data_path))


[nltk_data] Downloading package punkt to l:\Machine Learning
[nltk_data]     Programming\NLP_PipelineIntroductionWorkshop\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping tokenizers\punkt.zip.


[nltk_data] Downloading package punkt_tab to l:\Machine Learning
[nltk_data]     Programming\NLP_PipelineIntroductionWorkshop\nltk_data
[nltk_data]     ...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to l:\Machine Learning
[nltk_data]     Programming\NLP_PipelineIntroductionWorkshop\nltk_data
[nltk_data]     ...


[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to l:\Machine Learning
[nltk_data]     Programming\NLP_PipelineIntroductionWorkshop\nltk_data
[nltk_data]     ...


NLTK Data Paths: ['C:\\Users\\atat7/nltk_data', 'l:\\Machine Learning Programming\\NLP_PipelineIntroductionWorkshop\\venv\\nltk_data', 'l:\\Machine Learning Programming\\NLP_PipelineIntroductionWorkshop\\venv\\share\\nltk_data', 'l:\\Machine Learning Programming\\NLP_PipelineIntroductionWorkshop\\venv\\lib\\nltk_data', 'C:\\Users\\atat7\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'l:\\Machine Learning Programming\\NLP_PipelineIntroductionWorkshop\\nltk_data']
Contents of nltk_data: ['corpora', 'tokenizers']


Now, let's load the data and see what a raw document looks like.

In [44]:
# Load the dataset
from sklearn.datasets import fetch_20newsgroups
newsgroups_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

# Let's inspect a single document
print("--- RAW DOCUMENT ---")
sample_doc = newsgroups_data.data[12]
print(sample_doc)

# Let's get some basic stats
all_text = ' '.join(newsgroups_data.data)
words = all_text.split()
unique_words = set(words)

print(f"\n--- INITIAL STATS ---")
print(f"Number of documents: {len(newsgroups_data.data)}")
print(f"Total words (approx.): {len(words)}")
print(f"Total unique words (vocab size): {len(unique_words)}")

--- RAW DOCUMENT ---
930418

Do what thou wilt shall be the whole of the Law. [Honestly.]
The word of Sin is Restriction. [Would I kid you?]


Does one man's words encompass the majestic vision of thousands
of individuals?  Quoting a man is not the same as quoting the
Order.  Taken out of context, words can be interpreted much
differently than had one applied them within the confines of
their original expression.

I think this is the case regarding Hymenaeus Beta, Frater Superior 
of the Order to which I belong.  When he included that bit
from Merlinus X' he did us all a service.  He showed us the extremes
to which Order members have been known to go in their fervor.
I have little knowledge regarding Reuss' background, but surely
he was an unusual man, and he was an important force in the Order 
for many years.

Yet as people change so do Orders change, and while we look back
so carefully at the dirty laundry of O.T.O. remember that this is
only the surface skim and that many perspecti

You'll notice our vocabulary is massive! This is the complexity we need to reduce.

### 3. The NLP Pipeline: From Chaos to Structure

We will now build the core components of our pipeline. Each step feeds its output to the next, progressively refining the text.

#### Part 1: Tokenization - Creating the Building Blocks

Concept: Tokenization is the process of breaking down a stream of text into individual units, or tokens. These tokens are the fundamental building blocks for any further analysis. Usually, tokens are words, but they can also be sentences or characters.

Implementation: A Simple Tokenizer
We can start with a basic algorithm that splits text by spaces and removes punctuation.

In [45]:
import re

def simple_tokenizer(text):
  """
  A simple implementation of a word tokenizer.
  1. Splits text by non-word characters.
  2. Returns a list of word tokens.
  """
  # Use regular expression to find all sequences of word characters
  tokens = re.findall(r'\b\w+\b', text)
  return tokens

# Let's tokenize our sample document
raw_tokens = simple_tokenizer(sample_doc)
print("--- TOKENIZATION ---")
print(raw_tokens)

--- TOKENIZATION ---
['930418', 'Do', 'what', 'thou', 'wilt', 'shall', 'be', 'the', 'whole', 'of', 'the', 'Law', 'Honestly', 'The', 'word', 'of', 'Sin', 'is', 'Restriction', 'Would', 'I', 'kid', 'you', 'Does', 'one', 'man', 's', 'words', 'encompass', 'the', 'majestic', 'vision', 'of', 'thousands', 'of', 'individuals', 'Quoting', 'a', 'man', 'is', 'not', 'the', 'same', 'as', 'quoting', 'the', 'Order', 'Taken', 'out', 'of', 'context', 'words', 'can', 'be', 'interpreted', 'much', 'differently', 'than', 'had', 'one', 'applied', 'them', 'within', 'the', 'confines', 'of', 'their', 'original', 'expression', 'I', 'think', 'this', 'is', 'the', 'case', 'regarding', 'Hymenaeus', 'Beta', 'Frater', 'Superior', 'of', 'the', 'Order', 'to', 'which', 'I', 'belong', 'When', 'he', 'included', 'that', 'bit', 'from', 'Merlinus', 'X', 'he', 'did', 'us', 'all', 'a', 'service', 'He', 'showed', 'us', 'the', 'extremes', 'to', 'which', 'Order', 'members', 'have', 'been', 'known', 'to', 'go', 'in', 'their', 'ferv

> Talking Point: "Tokenization is the first and most critical step; it defines the discrete units of meaning—our 'words'—that the machine will learn from."

#### Part 2: Normalization & Stop Words Removal - Cleaning the Noise

Concept: Normalization is about standardizing tokens to treat different forms of a word as the same. The most common step is converting all text to lowercase. Why? Because we don't want the model to think "Apple" (the company) and "apple" (the fruit) are different purely based on capitalization if the context doesn't require it.

Concept: Stop words are extremely common words like "the," "a," "is," "in." They add grammatical structure for humans but often add little semantic value for a machine. Removing them helps the model focus on the words that carry the most meaning.

Implementation: Lowercasing and Filtering

In [46]:
from nltk.corpus import stopwords

def normalize_and_remove_stops(tokens):
    """
    1. Converts all tokens to lowercase (Normalization).
    2. Removes common English stop words.
    """
    # 1. Normalization: Convert to lowercase
    normalized_tokens = [token.lower() for token in tokens]

    # 2. Stop Words Removal
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in normalized_tokens if token not in stop_words]
    return filtered_tokens

# Process our tokens
cleaned_tokens = normalize_and_remove_stops(raw_tokens)
print("--- NORMALIZATION & STOP WORDS REMOVAL ---")
print(cleaned_tokens)

--- NORMALIZATION & STOP WORDS REMOVAL ---
['930418', 'thou', 'wilt', 'shall', 'whole', 'law', 'honestly', 'word', 'sin', 'restriction', 'would', 'kid', 'one', 'man', 'words', 'encompass', 'majestic', 'vision', 'thousands', 'individuals', 'quoting', 'man', 'quoting', 'order', 'taken', 'context', 'words', 'interpreted', 'much', 'differently', 'one', 'applied', 'within', 'confines', 'original', 'expression', 'think', 'case', 'regarding', 'hymenaeus', 'beta', 'frater', 'superior', 'order', 'belong', 'included', 'bit', 'merlinus', 'x', 'us', 'service', 'showed', 'us', 'extremes', 'order', 'members', 'known', 'go', 'fervor', 'little', 'knowledge', 'regarding', 'reuss', 'background', 'surely', 'unusual', 'man', 'important', 'force', 'order', 'many', 'years', 'yet', 'people', 'change', 'orders', 'change', 'look', 'back', 'carefully', 'dirty', 'laundry', 'remember', 'surface', 'skim', 'many', 'perspectives', 'encompassed', 'extend', 'beyond', 'one', 'individual', 'hope', 'show', 'much', 'room'

> Talking Point (Normalization): "By converting every token to lowercase, we prevent the model from treating the same word differently, drastically reducing the vocabulary size it needs to handle."

> Talking Point (Stop Words): "Removing stop words is like filtering out the static; it lets the model focus on the high-signal words that define the document's topic."

#### Part 3: Stemming - Finding the Root

Concept: Stemming is a process of reducing a word to its root or stem. It does this by applying a crude set of rules to chop off common prefixes or suffixes (affixes). For example, "running," "runs," and "ran" might all be stemmed to the base form "run." This process, known as affix stripping, is heuristic and can sometimes result in non-dictionary words (e.g., "studies" becomes "studi"). However, it's fast and effective for collapsing vocabulary.

Implementation: The Porter Stemmer
We will use the famous Porter Stemming algorithm, available in NLTK.

In [47]:
from nltk.stem import PorterStemmer

def stem_tokens(tokens):
    """
    Applies the Porter Stemming algorithm to a list of tokens.
    This is a form of affix stripping.
    """
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens

# Apply stemming to our cleaned tokens
final_tokens = stem_tokens(cleaned_tokens)
print("--- STEMMING (AFFIX STRIPPING) ---")
print(final_tokens)

--- STEMMING (AFFIX STRIPPING) ---
['930418', 'thou', 'wilt', 'shall', 'whole', 'law', 'honestli', 'word', 'sin', 'restrict', 'would', 'kid', 'one', 'man', 'word', 'encompass', 'majest', 'vision', 'thousand', 'individu', 'quot', 'man', 'quot', 'order', 'taken', 'context', 'word', 'interpret', 'much', 'differ', 'one', 'appli', 'within', 'confin', 'origin', 'express', 'think', 'case', 'regard', 'hymenaeu', 'beta', 'frater', 'superior', 'order', 'belong', 'includ', 'bit', 'merlinu', 'x', 'us', 'servic', 'show', 'us', 'extrem', 'order', 'member', 'known', 'go', 'fervor', 'littl', 'knowledg', 'regard', 'reuss', 'background', 'sure', 'unusu', 'man', 'import', 'forc', 'order', 'mani', 'year', 'yet', 'peopl', 'chang', 'order', 'chang', 'look', 'back', 'care', 'dirti', 'laundri', 'rememb', 'surfac', 'skim', 'mani', 'perspect', 'encompass', 'extend', 'beyond', 'one', 'individu', 'hope', 'show', 'much', 'room', 'differ', 'opinion', 'within', 'order', 'perhap', 'test', 'limit', 'let', 'us', 'exami

> Talking Point (Stemming/Affix Stripping): "Stemming aggressively strips word endings to group related terms; it’s a powerful but blunt tool for consolidating our vocabulary down to its core concepts."

#### Part 4: The Final Result & The Big Picture

Let's put it all together and see the transformation.

In [48]:
# The full pipeline
def nlp_pipeline(text):
    tokens = simple_tokenizer(text)
    cleaned_tokens = normalize_and_remove_stops(tokens)
    final_tokens = stem_tokens(cleaned_tokens)
    return final_tokens

# Before
print("--- BEFORE PIPELINE ---")
print(sample_doc)

print("\n" + "="*50 + "\n")

# After
print("--- AFTER PIPELINE ---")
processed_doc = nlp_pipeline(sample_doc)
print(processed_doc)

# Let's re-evaluate our vocabulary size on the entire dataset
all_processed_tokens = nlp_pipeline(' '.join(newsgroups_data.data))
final_vocab_size = len(set(all_processed_tokens))

print(f"\n--- FINAL STATS ---")
print(f"Initial unique words: {len(unique_words)}")
print(f"Final unique words (vocab size): {final_vocab_size}")
print(f"Vocabulary Reduction: {((len(unique_words) - final_vocab_size) / len(unique_words)) * 100:.2f}%")

--- BEFORE PIPELINE ---
930418

Do what thou wilt shall be the whole of the Law. [Honestly.]
The word of Sin is Restriction. [Would I kid you?]


Does one man's words encompass the majestic vision of thousands
of individuals?  Quoting a man is not the same as quoting the
Order.  Taken out of context, words can be interpreted much
differently than had one applied them within the confines of
their original expression.

I think this is the case regarding Hymenaeus Beta, Frater Superior 
of the Order to which I belong.  When he included that bit
from Merlinus X' he did us all a service.  He showed us the extremes
to which Order members have been known to go in their fervor.
I have little knowledge regarding Reuss' background, but surely
he was an unusual man, and he was an important force in the Order 
for many years.

Yet as people change so do Orders change, and while we look back
so carefully at the dirty laundry of O.T.O. remember that this is
only the surface skim and that many perspe

### Part 5: Conclusion 

As you can see, we've taken unstructured, messy human language and converted it into a clean, standardized list of meaningful tokens. We've dramatically reduced the vocabulary size, making it much more efficient for an AI model to process.

This processed data is now ready to be converted into numerical vectors—a process called vectorization (e.g., TF-IDF or Word2Vec)—which is the final step before feeding it into a machine learning model to perform tasks like text classification, sentiment analysis, or even to train a foundational model like BARD.

### Part 6: Fun Activity: Ask BERT a Question!

So far, we've built an NLP pipeline from scratch. This is a fundamental skill, and it shows you the deliberate, step-by-step process of making text useful for machines.

Now, let's jump to the forefront of NLP and use a massive, pre-trained model: BERT (Bidirectional Encoder Representations from Transformers).

What is BERT?
Think of the pipeline we just built. Its goal was to simplify and standardize words. A model like BERT does something far more complex. During its "pre-training" on nearly the entire internet, it learned not just words, but the relationships between words.

Watch this video as an introduction to Transfoemrs to help you understand what BERT is, and where it came from:

[Transformer models and BERT model: Overview](http://www.youtube.com/watch?v=t45S_MwAcOw)


The key is its bidirectional nature. Unlike older models that read text left-to-right, BERT reads the entire sentence at once. This allows it to understand context. For example, it knows that "bank" in "river bank" is different from "bank" in "bank account." The pipeline we built would stem both to "bank" and lose that contextual meaning.

We can use the power of these pre-trained models for specific tasks without needing to train them ourselves. Let's try one of the most popular tasks: Extractive Question Answering.

The Goal: We will give the model a paragraph of text (the context) and a question. The model's job is to find and extract the exact span of text from the context that answers our question.

#### Implementation: The Hugging Face Pipeline 🚀

The transformers library from Hugging Face makes using models like BERT incredibly simple with a tool called pipeline. It handles all the complex tokenization and processing behind the scenes, so we can focus on the results.

In [49]:
#
# Prerequisite: Make sure you have transformers installed!
# Prerequisite: Install the necessary libraries
# The 'transformers' library requires either PyTorch or TensorFlow.
# We'll install torch here.
#
# !pip install transformers torch
#
from transformers import pipeline
import textwrap # Used for nice printing of the context

# 1. Create a Question-Answering pipeline
# This will download a default BERT-based model fine-tuned for this task.
# No authentication is needed!
qa_pipeline = pipeline("question-answering")

# 2. Define the context
# This is the body of text the model will read to find the answer.
context = """
BERT, which stands for Bidirectional Encoder Representations from Transformers,
is a large language model developed by Google. It was a major breakthrough in
the field of Natural Language Processing because it was the first model to look
at the full context of a word by examining the words that come before and after it.
This bidirectional capability allows it to grasp complex nuances in language.
The model was pre-trained on a massive amount of text data from sources like
Wikipedia and the BooksCorpus.
"""

# 3. Ask a question!
# The answer to this question must be present in the context above.
question = "What does BERT stand for?"

# 4. Get the answer
result = qa_pipeline(question=question, context=context)

# Let's print the results nicely
print("--- BERT Question-Answering ---")
print(f"❓ Question: {question}")
print(f"✅ Answer: '{result['answer']}'")
print(f"Confidence Score: {result['score']:.4f}")

# --- Now, it's your turn! ---
#
# Try asking different questions. For example:
# - "Who developed BERT?"
# - "What capability allows BERT to grasp nuances?"
# - "What was the model pre-trained on?"
#
# You can even change the context entirely to a paragraph about a
# topic you find interesting and ask questions about it!

print("\n--- Try it yourself! ---")
# Example of another question
my_question = "Who developed BERT?"
my_result = qa_pipeline(question=my_question, context=context)
my_question2 = "What capability allows BERT to grasp nuances?"
my_result2 = qa_pipeline(question=my_question2, context=context)
my_question3 = "What was the model pre-trained on?"
my_result3 = qa_pipeline(question=my_question3, context=context)

print(f"❓ My Question: {my_question}")
print(f"✅ Answer: '{my_result['answer']}' with score {my_result['score']:.4f}")
print(f"❓ My Question2: {my_question2}")
print(f"✅ Answer2: '{my_result2['answer']}' with score {my_result2['score']:.4f}")
print(f"❓ My Question3: {my_question3}")
print(f"✅ Answer3: '{my_result3['answer']}' with score {my_result3['score']:.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


--- BERT Question-Answering ---
❓ Question: What does BERT stand for?
✅ Answer: 'Bidirectional Encoder Representations from Transformers'
Confidence Score: 0.7050

--- Try it yourself! ---
❓ My Question: Who developed BERT?
✅ Answer: 'Google' with score 0.9778
❓ My Question2: What capability allows BERT to grasp nuances?
✅ Answer2: 'bidirectional' with score 0.7288
❓ My Question3: What was the model pre-trained on?
✅ Answer3: 'a massive amount of text data' with score 0.1840


⚠️ A Quick Word on Model Limitations
It's important to understand that the BERT model we're using is not a general-purpose chatbot. It has been fine-tuned for a very specific task: extractive question answering.

This means its only job is to find and extract the exact span of text from the context that best answers the question. It does not understand abstract concepts, perform reasoning, or generate new sentences.

Watch what happens when we ask it about a joke.

In [50]:
# Example of another question
my_question = "Can BERT tell jokes?"
my_result = qa_pipeline(question=my_question, context=context)

print(f"❓ My Question: {my_question}")
print(f"✅ Answer: '{my_result['answer']}' with score {my_result['score']:.4f}")

❓ My Question: Can BERT tell jokes?
✅ Answer: 'Bidirectional Encoder Representations from Transformers' with score 0.0830


In [51]:
# The context contains a simple joke
joke_context = "I told my computer I needed a break, and now it won’t stop sending me Kit-Kat ads."

# The question requires understanding humor, not just extracting text
joke_question = "Why is this joke funny?"

# Let's see how the extractive model handles it
joke_result = qa_pipeline(question=joke_question, context=joke_context)

print(f"❓ Question: {joke_question}")
print(f"✅ Answer: '{joke_result['answer']}'")
print(f"Confidence Score: {joke_result['score']:.4f}")

❓ Question: Why is this joke funny?
✅ Answer: 'I needed a break'
Confidence Score: 0.0999


As you can see, the model fails. It extracts a piece of the sentence but completely misses the pun and the humorous concept. The low confidence score also shows it is struggling.

This model will generally fail at questions that require:

Reasoning or Synthesis: It can't combine multiple pieces of information.

"Yes/No" Answers: Unless the words "yes" or "no" are explicitly in the text.

Abstract Concepts: Like humor, sarcasm, or summarizing the main idea.

This is a key difference between an extractive model like this one and a generative AI like BARD, which is designed to understand these nuances and generate new, explanatory text.

### Part 7: Challenges

You have now seen a complete introductory NLP pipeline in action. But real NLP systems are rarely “finished.”  
There is almost always a missing piece, a trade-off, or a design decision that could be improved.

In this section, you will tackle **four beginner-friendly challenges** that push the current notebook a little further.  
Each challenge asks you to investigate a limitation, improve part of the pipeline, or test whether a different idea works better.

These are not just coding exercises — they are the kind of questions people ask when building real machine learning systems:

- Are we removing useful information by accident?
- Are we simplifying words too aggressively?
- How can we turn processed text into features a model could actually use?
- How does tokenization affect cost when we use commercial LLMs?

Take your time, experiment, and discuss what you notice.

#### Challenge 1: Are We Throwing Away Important Words?

Our current pipeline removes common stop words such as **"the"**, **"is"**, and **"and"**.  
That usually helps reduce noise, but it can also create a problem.

Some so-called “small” words actually carry **big meaning**.  
For example, the sentence:

> “This movie is **not** good.”

changes completely if we remove the word **not**.

Your challenge is to improve the stop-word removal step so that it keeps a few important **negation words** such as:

- `not`
- `no`
- `never`

Then test the result on a short sentence and see how the meaning changes when these words are preserved.

**Goal:** make the pipeline a little smarter by keeping words that are small, but important.

In [52]:
# Challenge 1 Scaffold:
# Modify stop-word removal so that important negation words are preserved.

test_sentence = "This movie is not good, and I would never recommend it to anyone."

def normalize_and_remove_stops_keep_negation(tokens):
    """
    1. Converts tokens to lowercase.
    2. Removes many common stop words.
    3. Keeps important negation words such as 'not', 'no', and 'never'.
    """
    # TODO: Convert each token to lowercase
    normalized_tokens = [token.lower() for token in tokens]
    # stopwords from nltk
    stop_words = set(stopwords.words('english'))

    # TODO: Remove a few negation words from the stop-word list
    # Hint: look up set removal methods such as discard() or difference_update()
    negation_words = {"not", "no", "never"}

    # TODO: update stop_words so the negation words are NOT removed

    stop_words.difference_update(negation_words)
    
    # TODO: Filter the tokens using the updated stop_words set
    filtered_tokens = [token for token in normalized_tokens if token not in stop_words]

    return filtered_tokens

# Try the original pipeline logic on the test sentence
original_tokens = simple_tokenizer(test_sentence)
original_cleaned = normalize_and_remove_stops(original_tokens)

# Try your improved version
improved_cleaned = normalize_and_remove_stops_keep_negation(original_tokens)

print("--- Original cleaned tokens ---")
print(original_cleaned)

print("\n--- Improved cleaned tokens (keep negation) ---")
print(improved_cleaned)

# TODO:
# 1. Compare the two outputs.
# 2. Identify which words were preserved in the improved version.
# 3. Explain why that matters for meaning.

--- Original cleaned tokens ---
['movie', 'good', 'would', 'never', 'recommend', 'anyone']

--- Improved cleaned tokens (keep negation) ---
['movie', 'not', 'good', 'would', 'never', 'recommend', 'anyone']


**Talking Points — Challenge 1**

Write your observations below:

- What words disappeared in the original version?
- Which negation words did your improved version preserve?
- How could removing words like `not` or `never` confuse a sentiment analysis model?
- Which line(s) of code were most important to solving this challenge?

> Your notes here:
>
> - **What words disappeared in the original version?**
>
>   In the first version, the word "not" was deleted.
>
> - **Which negation words did your improved version preserve?**
>
>   The new version kept the important word "not".
>
> - **How could removing words like `not` or `never` confuse a sentiment analysis model?**
>
>   If we delete "not", the AI thinks "not good" means "good". This makes the AI believe a bad movie is a good movie.
>
> - **Which line(s) of code were most important to solving this challenge?**
>
>   The most important line was `stop_words.difference_update(negation_words)`. This line tells the computer to keep the negation words.

#### Challenge 2: Is Stemming Too Aggressive?

Our notebook currently uses **stemming** to reduce words to a simpler form.  
That is useful, but stemming can sometimes be a little rough.

For example:

- `studies` might become `studi`
- `running` might become `run`
- `better` may not behave the way you expect

This raises a good beginner question:

> Is stemming always the best choice?

A more language-aware alternative is **lemmatization**, which tries to convert words into a proper dictionary base form.

Your challenge is to compare **stemming** and **lemmatization** on a small list of words and reflect on which output seems easier for humans to interpret.

**Goal:** explore the trade-off between a fast, simple method and a more linguistically meaningful method.

In [53]:
# Challenge 2 Scaffold:
# Compare stemming and lemmatization on a small set of example words.

# If needed, uncomment the following line once:
# import nltk
# nltk.download('wordnet')
# nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer
import time

example_words = ["running", "studies", "better", "cars", "wolves", "playing"]
large_test_words = example_words * 10000

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = []
lemmatized_words = []

for word in example_words:
    # TODO: Stem the word and append the result to stemmed_words
    # Hint: use stemmer.stem(...)
    stemmed_words.append(stemmer.stem(word))

for word in example_words:
    # TODO: Lemmatize the word and append the result to lemmatized_words
    # Hint: use lemmatizer.lemmatize(...)
    lemmatized_words.append(lemmatizer.lemmatize(word))

# test Stemming speed
start_stem = time.time()
for word in large_test_words:
    stemmer.stem(word)
end_stem = time.time()

# test Lemmatization speed
start_lem = time.time()
for word in large_test_words:
    lemmatizer.lemmatize(word)
end_lem = time.time()

print("--- Original Words ---")
print(example_words)

print("\n--- Stemmed Words ---")
print(stemmed_words)
print("\n--- Stemming time ---")
print(f"Stemming time: {end_stem - start_stem:.4f} seconds")

print("\n--- Lemmatized Words ---")
print(lemmatized_words)
print("\n--- Lemmatization time ---")
print(f"Lemmatization time: {end_lem - start_lem:.4f} seconds")

# TODO:
# 1. Compare the stemmed and lemmatized results.
# 2. Which output looks more like normal English words?
# 3. Which method do you think would be easier to explain to a non-technical user?

--- Original Words ---
['running', 'studies', 'better', 'cars', 'wolves', 'playing']

--- Stemmed Words ---
['run', 'studi', 'better', 'car', 'wolv', 'play']

--- Stemming time ---
Stemming time: 1.1701 seconds

--- Lemmatized Words ---
['running', 'study', 'better', 'car', 'wolf', 'playing']

--- Lemmatization time ---
Lemmatization time: 0.2671 seconds


**Talking Points — Challenge 2**

Write your observations below:

- Which words looked strange after stemming?
- Which words looked more natural after lemmatization?
- Why might a data scientist still choose stemming even if the output looks less polished?
- What did this challenge teach you about trade-offs in NLP?

> Your notes here:
>
> - **Which words looked strange after stemming?**
>
>   The words "studi" and "wolv" looked strange. These are not correct English words because `Stemmed` just cut the endings.
>   
> - **Which words looked more natural after lemmatization?**
>
>   The words "study" and "wolf" looked natural.
>   
> - **Why might a data scientist still choose stemming even if the output looks less polished?**
>
>   In theory, stemming is faster for very big data. But in my small test, Lemmatization was quicker!
>   
> - **What did this challenge teach you about trade-offs in NLP?**
>
>   In my test, Stemming took 1.20s and Lemmatization took 0.27s. I think this happened because the computer "remembered" (cached) the dictionary words, or because my list of words was very small.I learned that theory and practice can be different. We should always test our code to see the real speed.


#### Challenge 3: How Do We Turn Cleaned Text into Features for Machine Learning?

So far, we have cleaned text and produced processed tokens.  
That is an important step, but machine learning models still need **numbers**, not just words.

One common beginner-friendly solution is a **Bag-of-Words** representation.  
The idea is simple:

- count how often words appear
- choose the most common terms
- represent each document using those counts

This is one of the classic bridges between **NLP** and **Machine Learning**.

Your challenge is to build a small Bag-of-Words feature view from the processed dataset and inspect the most frequent tokens.

**Goal:** connect text preprocessing to the kind of numerical features a model could actually learn from.

In [54]:
# Challenge 3 Scaffold:
# Build a simple Bag-of-Words style summary from processed text.

from collections import Counter

# Reuse a small subset of documents so the output stays manageable
small_corpus = newsgroups_data.data[:50]

processed_documents = []

for doc in small_corpus:
    # TODO: Run the existing nlp_pipeline on each document
    # and store the processed tokens in processed_documents
    processed_doc = nlp_pipeline(doc)
    processed_documents.append(processed_doc)

# Flatten the list of token lists into one big list of tokens
all_tokens = []

for token_list in processed_documents:
    # TODO: Extend all_tokens with the tokens from each processed document
    all_tokens.extend(token_list)

token_counts = Counter(all_tokens)

# TODO: Display the 15 most common processed tokens
top_15 = token_counts.most_common(15)

print("--- Top 15 Tokens in the Small Corpus ---")
print(top_15)

# Optional extension:
# TODO: Create a tiny Bag-of-Words vector for one processed document.
# Steps:
# 1. Choose the top 5 most common tokens from top_15
# 2. Count how many times each of those 5 tokens appears in one document
# 3. Store those counts in a dictionary or list

# Example starter:
selected_doc = processed_documents[0]
vocabulary = [token for token, count in top_15[:20]]
bow_vector = {}

# TODO: Fill bow_vector so that each vocabulary word maps to its count in selected_doc
for word in vocabulary:
    bow_vector[word] = selected_doc.count(word)

print("\n--- Example Bag-of-Words Vector ---")
print(bow_vector)

selected_doc = processed_documents[12] 

# Which article contains the word 'order
for i, doc in enumerate(processed_documents):
    if 'order' in doc:
        print("\n--- Which article contains the word 'order'? ---")
        print(f"The article {i} contains 'order'!")
        selected_doc = processed_documents[i]
        break

--- Top 15 Tokens in the Small Corpus ---
[('det', 45), ('nyr', 40), ('tor', 39), ('bo', 37), ('mtl', 33), ('la', 30), ('chi', 30), ('order', 26), ('one', 25), ('pit', 24), ('time', 23), ('use', 22), ('edm', 22), ('phi', 21), ('1', 20)]

--- Example Bag-of-Words Vector ---
{'det': 0, 'nyr': 0, 'tor': 0, 'bo': 0, 'mtl': 0, 'la': 0, 'chi': 0, 'order': 0, 'one': 0, 'pit': 0, 'time': 0, 'use': 0, 'edm': 0, 'phi': 0, '1': 0}

--- Which article contains the word 'order'? ---
The article 4 contains 'order'!


**Talking Points — Challenge 3**

Write your observations below:

- Which processed words appeared most often?
- Did the most common words seem meaningful, surprising, or too generic?
- How does a Bag-of-Words representation help a machine learning model?
- What information is lost when we reduce a document to word counts?

> Your notes here:
>
> - **Which processed words appeared most often?**
>
>   The most common words are city codes like "det", "nyr", and "tor". Words like "order" and "one" are also in the top list.
>
> - **Did the most common words seem meaningful, surprising, or too generic?**
>
>   It was surprising because I saw many sports codes. My first document had zero matches, but I found that article 4 has the word "order".
>
> - **How does a Bag-of-Words representation help a machine learning model?**
>
>   It changes text into numbers. The model uses these numbers to find which words are important for each topic.
>
> - **What information is lost when we reduce a document to word counts?**
>
>   We lose the grammar and the word order. The computer knows the "counts" but it does not understand the "story".
>

#### Challenge 4: Why Does Tokenization Matter for LLM Billing?

When we use a commercial Large Language Model, we are usually **not billed per prompt** or per paragraph.  
We are billed by **tokens**.

A token is a chunk of text. Sometimes it is a whole word, sometimes part of a word, and sometimes punctuation.  
That means two prompts that look very similar to a human can still produce **different token counts, costs, and response times**.

This makes tokenization more than just a preprocessing step.  
It becomes part of a real engineering decision:

- How much text should we send?
- Should we remove stop words?
- Should we simplify words first?
- Does a Bag-of-Words view reduce meaning too much?
- How much extra cost comes from the **prompt** compared with the **model's answer**?

For this reflection challenge, imagine you are preparing a long educational prompt for a commercial LLM.  
Use the prompt below as your sample text, then explore how the token count changes if you:

1. remove stop words  
2. apply stemming or lemmatization  
3. convert the text into a Bag-of-Words style view  

You may also compare your observations with external tools such as:

- `https://pricepertoken.com/`
- `https://pricepertoken.com/token-counter`

A real example shared for this workshop showed that two models could receive almost the same prompt, yet differ a lot in **completion length, latency, and cost**.  
That is a useful reminder: **tokenization affects billing, but the model's response style matters too.**

**Goal:** reflect on how preprocessing choices may reduce token counts, and also think critically about what is lost when we compress a prompt too aggressively.


In [55]:
# Challenge 4 Scaffold:
# Reflect on how tokenization affects the cost of using a commercial LLM.

from collections import Counter
import re

from matplotlib import text

# You can replace this with any long prompt you want to analyze.
workshop_prompt = """
I am a professor of Machine Learning at the college level. You are an experienced
Machine Learning Engineer and Data Scientist. I am developing a workshop on time series
using Jupyter Notebooks as a project to help train my students in an active learning environment.

In this case, I want them to learn Data and Time Series Analysis, the Moving Averages calculation,
Decomposition Models, how to Deseasonalize a Time Series, Autoregressive Models, ARMA Models,
ARIMA Models, the Augmented Dickey-Fuller test, and Machine Learning application design patterns,
all simultaneously.

The workshop is based on active learning and requires them to read, brainstorm, write code,
reflect on model performance, and compare results with their peers.
""".strip()

# A very simple tokenizer for classroom experimentation.
# It is NOT the same as a commercial tokenizer, but it gives us a useful starting point.
def simple_tokenize(text):
    # TODO: Split the text into lowercase tokens.
    # Hint: you can use a regular expression such as r"\b\w+\b"
    tokens = re.findall(r"\b\w+\b", text.lower())
    return tokens

def remove_stop_words(tokens):
    stop_words = set(stopwords.words("english"))

    # TODO:
    # Remove stop words from the token list.
    # You may also choose to KEEP some words if you think they matter for meaning.
    filtered_tokens = [t for t in tokens if t not in stop_words]
    return filtered_tokens

def apply_stemming(tokens):
    stemmer = PorterStemmer()

    # TODO:
    # Apply stemming to each token and return the transformed list.
    stemmed_tokens = [stemmer.stem(t) for t in tokens]
    return stemmed_tokens

def bag_of_words_view(tokens):
    # TODO:
    # Build a Bag-of-Words representation using Counter.
    # Return the counts of each token.
    bow = Counter(tokens)
    return bow

# --- Baseline ---
baseline_tokens = simple_tokenize(workshop_prompt)
print("Baseline token count:", len(baseline_tokens))
print("First 25 baseline tokens:", baseline_tokens[:25])

# --- Version A: Stop-word removal ---
tokens_no_stops = remove_stop_words(baseline_tokens)
print("\nToken count after stop-word removal:", len(tokens_no_stops))

# --- Version B: Stemming ---
stemmed_tokens = apply_stemming(tokens_no_stops)
print("Token count after stemming:", len(stemmed_tokens))

# --- Version C: Bag-of-Words ---
bow_counts = bag_of_words_view(stemmed_tokens)

# TODO:
# Inspect the 15 most common tokens in the Bag-of-Words representation.
top_15 = bow_counts.most_common(15)
print("\nTop 15 Bag-of-Words terms:")
print(top_15)

# Optional extension:
# Estimate a hypothetical cost.
# Replace the rate below with a price you want to test, such as cost per 1K tokens.
price_per_1k_tokens = 0.008

# TODO:
# Estimate the baseline prompt cost and the processed prompt cost.
# Hint:
# baseline_cost = len(baseline_tokens) / 1000 * price_per_1k_tokens
# processed_cost = len(stemmed_tokens) / 1000 * price_per_1k_tokens
baseline_cost = (len(baseline_tokens) / 1000) * price_per_1k_tokens
processed_cost = (len(stemmed_tokens) / 1000) * price_per_1k_tokens

print("\nEstimated baseline prompt cost:", baseline_cost)
print("Estimated processed prompt cost:", processed_cost)

# Reflection prompt:
# TODO:
# In 2-3 sentences, explain which preprocessing step reduced the token count the most.
# Then explain whether that reduction would still preserve enough meaning for a real LLM task.


Baseline token count: 114
First 25 baseline tokens: ['i', 'am', 'a', 'professor', 'of', 'machine', 'learning', 'at', 'the', 'college', 'level', 'you', 'are', 'an', 'experienced', 'machine', 'learning', 'engineer', 'and', 'data', 'scientist', 'i', 'am', 'developing', 'a']

Token count after stop-word removal: 71
Token count after stemming: 71

Top 15 Bag-of-Words terms:
[('learn', 6), ('model', 5), ('machin', 3), ('time', 3), ('seri', 3), ('data', 2), ('workshop', 2), ('activ', 2), ('professor', 1), ('colleg', 1), ('level', 1), ('experienc', 1), ('engin', 1), ('scientist', 1), ('develop', 1)]

Estimated baseline prompt cost: 0.000912
Estimated processed prompt cost: 0.0005679999999999999


**Talking Points — Challenge 4**

Write your observations below:

- How many tokens did your baseline prompt contain?
- If stop words were removed, how much shorter did the prompt become?
- Did stemming or lemmatization reduce the token count much more, or mostly change the shape of the words?
- What happened when you converted the text into a Bag-of-Words view?
- Would you actually send the Bag-of-Words version to a commercial LLM? Why or why not?
- In your opinion, what matters more for billing in the example above: the prompt tokens or the completion tokens?
- Which preprocessing step saved the most tokens without damaging the meaning too much?

> Your notes here:
>
> - **How many tokens did your baseline prompt contain?**
>   
>   The baseline prompt had 114 tokens.
>
> - **If stop words were removed, how much shorter did the prompt become?**
>   
>   The prompt became much shorter. The count dropped from 114 to 71 tokens because we removed small words like "i", "am", and "a".
> 
> - **Did stemming or lemmatization reduce the token count much more, or mostly change the shape of the words?**
>   
>   In my test, stemming did not reduce the count. It only changed the shape of words, like changing "learning" to "learn". The token count stayed at 71.
> 
> - **What happened when you converted the text into a Bag-of-Words view?**
>   
>   The Bag-of-Words showed which words are the most important. For example, "learn" appeared 6 times and "model" appeared 5 times.
>
> - **Would you actually send the Bag-of-Words version to a commercial LLM? Why or why not?**
>   
>   No, I would not. A Bag-of-Words loses the word order. An LLM needs full sentences to understand the context and give a good answer.
>
> - **In your opinion, what matters more for billing in the example above: the prompt tokens or the completion tokens?**
>   
>   Both are important, but the model's response (completion) often costs more and can be longer than the prompt.
>
> - **Which preprocessing step saved the most tokens without damaging the meaning too much?**
>   
>   Removing stop words saved the most tokens. It made the prompt cheaper but kept the important information.
>

